In [1]:
PLANNER_PROMPT_TEMPLATE = """
You are a top AI planning expert. Your task is to decompose complex problems posed by users into an action plan consisting of multiple simple steps.
Please ensure that each step in the plan is an independent, executable subtask and is strictly arranged in logical order.
Your output must be a Python list, where each element is a string describing a subtask.

Question: {question}

Please strictly output your plan in the following format, with ```python and ``` as prefix and suffix being necessary:
```python
["Step 1", "Step 2", "Step 3", ...]
```
"""


In [ ]:
import ast
class Planner:
    def __init__(self,llm_client):
        self.llm = llm_client

    def plan(self,question:str):

        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)

        messages =[
            {"role":"user","content":prompt}
        ]

        response = self.llm.think(messages=messages)

        if not response:
          print("❌ Planner received no response from LLM.")
          return []

        try:

          plan_str = response.split("```python")[1].split("```")[0].strip()

          plan = ast.literal_eval(plan_str)

          return plan
        except Exception as e:
            print(f"❌ Error parsing plan:{e}")
            return []
        

        



'''''
question
  ↓
#insert into PLANNER_PROMPT_TEMPLATE
  ↓
get the complete prompt
  ↓
wrap it into messages
  ↓
send it to the LLM
  ↓
get the response
'''''


"''\nquestion\n  ↓\n#insert into PLANNER_PROMPT_TEMPLATE\n  ↓\nget the complete prompt\n  ↓\nwrap it into messages\n  ↓\nsend it to the LLM\n  ↓\nget the response\n"

In [ ]:
from llm_client import HelloAgentsLLM

llm_client =HelloAgentsLLM()

planner = Planner(llm_client)

result = planner.plan(
    "Research the latest developments in AI agents and summarize the main trends."
)

print("===== PLANNER RESPONSE =====")
print(result)

In [3]:
EXECUTOR_PROMPT_TEMPLATE = """
You are a top AI execution expert. Your task is to strictly follow the given plan and solve the problem step by step.
You will receive the original question, the complete plan, and the steps and results completed so far.
Please focus on solving the "current step" and only output the final answer for that step, without any additional explanations or dialogue.

# Original Question:
{question}

# Complete Plan:
{plan}

# Historical Steps and Results:
{history}

# Current Step:
{current_step}

Please only output the answer for the "current step":
"""


In [9]:
class Executor:
    def __init__(self,llm_client):
        self.llm = llm_client

    def execute(self,question:str,plan:list[str]):
        history = ""

        for i, step in enumerate(plan, start=1):

            print(f"Step {i}/{len(plan)}: {step}")

            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan = plan,
                history = history,
                current_step = step
            )

            messages = [
                {
                    "role":"user",
                "content":prompt
                }
            ]

            response = self.llm.think(messages=messages)

            if not response:
                print(f"❌ Step {i+1} failed: LLM returned no response.")
                return None

            history += f"""
            Step: {step}
            Result: {response}
            """
            print("===== CURRENT HISTORY =====")
            print(history)

        return response


In [ ]:
test_plan = [
    "Identify the main recent developments in AI agents.",
    "Analyze the common trends.",
    "Summarize the findings."
]

executor = Executor(llm_client)

result = executor.execute(
    question="Research the latest developments in AI agents and summarize the main trends.",
    plan=test_plan
)

print("===== FINAL RESULT =====")
print(result)

In [10]:
class PlanandSolveAgent:
    def __init__(self,llm_client):
        self.planner = Planner(llm_client)
        self.executor = Executor(llm_client)

    def run(self,question:str):

        plan = self.planner.plan(question)

        if not plan:
            print("Failed to generate plan.")
            return None
        

        final_result = self.executor.execute(
            question,
            plan
        )

        return final_result

In [11]:
from llm_client import HelloAgentsLLM

llm_client =HelloAgentsLLM()

agent = PlanandSolveAgent(llm_client)

result = agent.run(
    "A fruit store sold 15 apples on Monday. The number of apples sold on Tuesday was twice that of Monday. The number sold on Wednesday was 5 fewer than Tuesday. How many apples were sold in total over these three days?"
)

print("===== FINAL RESULT =====")
print(result)

MODEL: openrouter/free
BASE URL: https://openrouter.ai/api/v1
KEY PREFIX: sk-or-v1
🧠 Calling openrouter/free model...
✅ Large language model response successful:
```python
["Step 1: Calculate the number of apples sold on Tuesday (twice Monday's 15 apples)", "Step 2: Calculate the number of apples sold on Wednesday (5 fewer than Tuesday's amount)", "Step 3: Sum the apples sold on Monday, Tuesday, and Wednesday to get the total"]
```
Step 1/3: Step 1: Calculate the number of apples sold on Tuesday (twice Monday's 15 apples)
🧠 Calling openrouter/free model...
✅ Large language model response successful:
30
===== CURRENT HISTORY =====

            Step: Step 1: Calculate the number of apples sold on Tuesday (twice Monday's 15 apples)
            Result: 30
            
Step 2/3: Step 2: Calculate the number of apples sold on Wednesday (5 fewer than Tuesday's amount)
🧠 Calling openrouter/free model...
✅ Large language model response successful:
25
===== CURRENT HISTORY =====

            Ste